# 2.2. Data Preprocessing

D2L의 Data Preprocessing 장을 PyTorch 기준으로 변환해서 학습함.
실제 데이터는 처음부터 Tensor로 주어지지 않는다.
보통 CSV, Excel, DB, JSON 같은 형태로 저장되어 있고,
모델에 넣기 전에 pandas로 읽고 전처리한 뒤 Tensor로 변환해야 한다.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [9]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. CSV 파일 만들기

D2L에서는 작은 집값 데이터를 직접 만듬. house_tiny.csv
행은 하나의 집을 의미하고 열은 방개수, 지붕 종류, 가격등을 의미함

In [7]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)

data_file = os.path.join('..', 'data', 'house_tiny.csv')

with open(data_file, 'w') as f:
  f.write('''NumRooms, RoofType, Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000''') # NA는 값이 없다는 의미 (결측치)

## 2. pandas로 CSV 읽기

pandas는 표 형태 데이터를 다루는 대표적인 python 라이브러리임. D2L은 pd.read_csv를 사용해 읽음

In [ ]:
import pandas as pd

data = pd.read_csv(data_file)
print(data) # NA는 NaN으로 표시됨

   NumRooms  RoofType   Price
0       NaN       NaN  127500
1       2.0       NaN  106000
2       4.0     Slate  178100
3       NaN       NaN  140000
'NumRooms, RoofType, Price\nNA,NA,127500\n2,NA,106000\n4,Slate,178100\nNA,NA,140000'


## 3. 입력값과 정답값 분리

지도학습에서는 보통 데이터를 두 부분으로 나눈다.


>입력값 X → 모델이 보고 판단하는 정보  
>정답값 y → 모델이 맞혀야 하는 목표값

예제에서는 다음처럼 나눈다.
>입력값: NumRooms, RoofType  
>정답값: Price

In [11]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)
print(targets)

   NumRooms   RoofType_Slate   RoofType_nan
0       NaN            False           True
1       2.0            False           True
2       4.0             True          False
3       NaN            False           True
0    127500
1    106000
2    178100
3    140000
Name:  Price, dtype: int64


## 4. 결측치 처리

현실 데이터는 값이 비어있는 경우가 많다. D2L은 결측치 처리를 계속 마주치는 문제라고 설명한다.
결측치 처리 방법은 크게 두가지다

1. Imputation: 빈 값을 어떤 추정값으로 채우기
2. Deletion: 빈 값이 있는 행이나 열을 삭제하기

예를 들어 NumRooms가 비어 있으면 평균값으로 채울 수 있다. 2, 4 평균이 3이기 때문에 NaN을 3으로 채운다
NumRooms: [NaN, 2, 4, NaN] => [3, 2, 4, 3]

## 5. 범주형 데이터 처리

문제는 RoofType임. RoofType: [NaN, NaN, Slate, NaN] <br>
숫자가 아닌데 딥러닝 모델은 기본적으로 숫자 텐서를 받는다. 그래서 Slate같은걸 넣을 수 없다. <br>
그래서 범주형 데이터를 숫자로 바꿔야 한다. <br>
<br>
D2L은 get_dummies를 사용한다.<br>
원래 하나의 RoofType이라는 컬럼을 RoofType_Slate, RoofType_nan로 나눈다.<br>
<br>
각 행이 Slate면 RoofType_Slate=True, RoofType_nan=False가 된다.<br>
각 행이 결측치이면 RoofType_Slate=False, RoofType_nan=True가 된다.<br>
문자대신 0/1 형태가 되는 것이다. 나중에 텐서로 바꾸면 다음처럼 된다.

>False → 0  
>True  → 1

In [ ]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms   RoofType_Slate   RoofType_nan
0       NaN            False           True
1       2.0            False           True
2       4.0             True          False
3       NaN            False           True


## 6. Tensor로 변환

범주형 처리와 숫자형 결측치를 평균으로 모두 채우면 비어있던 NaN값이 모두 채워지게 된다.

이제 모든 값이 숫자처럼 표현될수 있다. 그래서 텐서로 바꿀수 있다.

In [14]:
inputs = inputs.fillna(inputs.mean())
print(inputs)

import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(targets.to_numpy(dtype=float))

X, y

   NumRooms   RoofType_Slate   RoofType_nan
0       3.0            False           True
1       2.0            False           True
2       4.0             True          False
3       3.0            False           True


(tensor([[3., 0., 1.],
         [2., 0., 1.],
         [4., 1., 0.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## 7. 오늘의 정리

- 실제 데이터는 처음부터 Tensor로 주어지지 않는다.
- 보통 CSV, DB, Excel, JSON 같은 형태로 존재한다.
- pandas는 표 형태 데이터를 읽고 전처리하는 데 많이 쓰인다.
- pd.read_csv()는 CSV 파일을 DataFrame으로 읽는다.
- DataFrame은 행과 열이 있는 표 형태 데이터다.
- 지도학습에서는 입력값 X와 정답값 y를 분리해야 한다.
- iloc는 정수 위치 기반으로 행과 열을 선택한다.
- CSV의 NA는 pandas에서 NaN으로 처리된다.
- NaN은 결측치다.
- 결측치 처리는 크게 채우기(imputation)와 삭제(deletion)가 있다.
- 숫자형 결측치는 평균값으로 채울 수 있다.
- 범주형 데이터는 그대로 Tensor로 넣을 수 없다.
- pd.get_dummies()는 범주형 데이터를 0/1 컬럼으로 바꾼다.
- dummy_na=True를 쓰면 NaN도 하나의 범주로 처리한다.
- 모든 값이 숫자로 바뀌면 torch.tensor()로 Tensor 변환이 가능하다.
- 최종 흐름은 CSV → pandas DataFrame → 전처리 → NumPy → PyTorch Tensor다.